<a href="https://colab.research.google.com/github/valceven/AliacSearchAlgo/blob/master/CNNActFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("wowza")

wowza


In [3]:
!git clone https://huggingface.co/datasets/seaurkin/facial_exrpressions

Cloning into 'facial_exrpressions'...
remote: Enumerating objects: 9364, done.
remote: Total 9364 (delta 0), reused 0 (delta 0), pack-reused 9364 (from 1)
Receiving objects: 100% (9364/9364), 1.50 MiB | 1.69 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Updating files: 100% (9365/9365), done.
Filtering content: 100% (9363/9363), 17.88 MiB | 244.00 KiB/s, done.


In [2]:
!ls ../content/

sample_data


In [4]:
import tensorflow
from tensorflow import keras
from keras.layers import Dense, GlobalAveragePooling2D, Conv2D, MaxPooling2D
from keras.optimizers import Adam
from keras.models import Model
from keras.applications import ResNet50V2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
import numpy as np
import pandas as pd
import requests
import os
from tqdm import tqdm
import torch
from PIL import Image
from sklearn.model_selection import train_test_split

In [5]:
dataset_path = "/content/facial_exrpressions/FACS"
labels = os.listdir(dataset_path)
print(labels)

['openmouth', 'kiss', 'smile', 'neutral', 'raisedbrows', 'frowning']


In [6]:
data = {"filename": [], "label": []};

for label in labels:
  label_path = os.path.join(dataset_path, label);
  if os.path.isdir(label_path):
    for file in os.listdir(label_path):
      if file.endswith((".jpg", ".png")):
        data["filename"].append(os.path.join(label_path, file))
        data["label"].append(label);

df = pd.DataFrame(data)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.head()

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["label"])
print(df["label"].unique())

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)}")
print(f"Val: {len(val_df)}")
print(f"Test: {len(test_df)}")

[1 5 0 4 3 2]
Train: 6554
Val: 1404
Test: 1405


In [7]:
train_df["label"] = train_df["label"].astype(str)
val_df["label"] = val_df["label"].astype(str)
test_df["label"] = test_df["label"].astype(str)

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
)

train_batches = datagen.flow_from_dataframe(
    train_df,
    x_col="filename",
    y_col="label",
    target_size=(224, 224),
    batch_size=32,
    class_mode="sparse",
    shuffle=True
)

valid_batches = datagen.flow_from_dataframe(
    val_df,
    x_col="filename",
    y_col="label",
    target_size=(224, 224),
    batch_size=32,
    class_mode="sparse",
    shuffle=True
)

test_batches = datagen.flow_from_dataframe(
    test_df,
    x_col="filename",
    y_col="label",
    target_size=(224, 224),
    batch_size=32,
    class_mode="sparse",
)

print(f"Train: {len(train_batches)}")
print(f"Val: {len(valid_batches)}")
print(f"Test: {len(test_batches)}")

Found 6554 validated image filenames belonging to 6 classes.
Found 1404 validated image filenames belonging to 6 classes.
Found 1405 validated image filenames belonging to 6 classes.
Train: 205
Val: 44
Test: 44


In [8]:
base_model = ResNet50V2(weights="imagenet",include_top=False, input_shape=(224, 224, 3))
base_model.trainable = True

model = base_model.output
model = GlobalAveragePooling2D()(model)
model = Dense(128, activation="relu")(model)
output_layer = Dense(len(labels), activation="softmax")(model)

model = Model(inputs=base_model.input, outputs=output_layer)
model.summary()

94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 224, 224, 3)    │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_pad (ZeroPadding2D) │ (None, 230, 230, 3)    │              0 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1_conv (Conv2D)       │ (None, 112, 112, 64)   │          9,472 │ conv1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pad (ZeroPadding2D) │ (None, 114, 114, 64)   │              0 │ conv1_conv[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ pool1_pool (MaxPooling2D) │ (None, 56, 56, 64)     │              0 │ pool1_pad[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_preact_bn    │ (None, 56, 56, 64)     │            256 │ pool1_pool[0][0]       │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_preact_relu  │ (None, 56, 56, 64)     │              0 │ conv2_block1_preact_b… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_conv       │ (None, 56, 56, 64)     │          4,096 │ conv2_block1_preact_r… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_bn         │ (None, 56, 56, 64)     │            256 │ conv2_block1_1_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_1_relu       │ (None, 56, 56, 64)     │              0 │ conv2_block1_1_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_pad        │ (None, 58, 58, 64)     │              0 │ conv2_block1_1_relu[0… │
│ (ZeroPadding2D)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_conv       │ (None, 56, 56, 64)     │         36,864 │ conv2_block1_2_pad[0]… │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_bn         │ (None, 56, 56, 64)     │            256 │ conv2_block1_2_conv[0… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_2_relu       │ (None, 56, 56, 64)     │              0 │ conv2_block1_2_bn[0][… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2_block1_0_conv       │ (None, 56, 56, 256)    │         16,640 │ conv2_block1_preact_r… │
│ (Conv2D)             

 Total params: 23,827,846 (90.90 MB)

 Trainable params: 23,782,406 (90.72 MB)

 Non-trainable params: 45,440 (177.50 KB)

In [12]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_batches, epochs=15, validation_data=valid_batches, batch_size=32)

Epoch 1/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 170s 612ms/step - accuracy: 0.9458 - loss: 0.1454 - val_accuracy: 0.9444 - val_loss: 0.1639
Epoch 2/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 107s 521ms/step - accuracy: 0.9631 - loss: 0.1051 - val_accuracy: 0.9416 - val_loss: 0.1667
Epoch 3/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 108s 524ms/step - accuracy: 0.9646 - loss: 0.0900 - val_accuracy: 0.9373 - val_loss: 0.1684
Epoch 4/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 106s 515ms/step - accuracy: 0.9632 - loss: 0.0971 - val_accuracy: 0.9466 - val_loss: 0.1880
Epoch 5/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 106s 515ms/step - accuracy: 0.9688 - loss: 0.0775 - val_accuracy: 0.9330 - val_loss: 0.1991
Epoch 6/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 106s 517ms/step - accuracy: 0.9694 - loss: 0.0757 - val_accuracy: 0.9487 - val_loss: 0.1783
Epoch 7/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 143s 522ms/step - accuracy: 0.9695 - loss: 0.0791 - val_accuracy: 0.9395 - val_loss: 0.1837
Epoch 8/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 106s 519ms/step - accuracy: 0.9722 -

In [13]:
model.evaluate(test_batches)

44/44 ━━━━━━━━━━━━━━━━━━━━ 18s 403ms/step - accuracy: 0.9519 - loss: 0.1401


[0.17102780938148499, 0.9423487782478333]

In [14]:
model.save("facial_expression_model.h5")

In [15]:
!ls

from google.colab import files
files.download("facial_expression_model.h5")

facial_expression_model.h5  facial_exrpressions  sample_data


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>